In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from datetime import date, datetime
import pandas as pd
from google.colab import drive

# Funções de extração de dados
def extrair_dados_produto(produto):
    try:
        nome = produto.find('div', class_='item-title').get_text(strip=True)
        link = produto.find('div', class_='item-title').a['href']
        preco = produto.find('span', class_='price').get_text(strip=True)
        descricao = produto.find('div', class_='desc').get_text(strip=True)
        imagem_url = produto.find('div', class_='thumb-wrapper').find('img')['src']

        # Salvar os dados em um diretorio
        dados_produto = {
            'Nome empresa': 'FOGOES SHOP',
            'Data de coleta': date.today().strftime('%Y-%m-%d'),
            'Imagem': imagem_url,
            'Nome': nome,
            'Preco': preco,
            'Link': link,
            'Descricao': descricao,
        }

        return dados_produto
    except Exception as e:
        print(f"\033[91mErro ao extrair dados do produto: {e}\033[0m")
        return None

def salvar_dados_csv(dados_produtos, category):
    try:
        df = pd.DataFrame(dados_produtos)

        # Criar a pasta 'output' se ainda não existir
        output_folder = "output"
        if not os.path.exists(output_folder):
            os.makedirs(output_folder)
            print(f"\033[92mA pasta {output_folder} foi criada.\033[0m")

        # Salvar o arquivo CSV na pasta 'output'
        file_path = os.path.join(output_folder, f"{category}.csv")
        df.to_csv(file_path, index=False)
        print(f"\033[92mDados da categoria '{category}' salvado com sucesso\033[0m")
        print()

        # Adicionar informações específicas
        print(f"\033[92mARQUIVO CRIADO {date.today()}\033[0m")

    except Exception as e:
        print(f"\033[91mErro ao salvar dados em CSV: {e}\033[0m")

def extrair_dados(url, category):
    try:
        print(f"Processando categoria '{category}' - URL: {url}")
        dados_produtos = []

        # Iteramos sobre todas as páginas
        page_num = 1
        while True:
            page_url = f"{url}/page/{page_num}/" if page_num > 1 else url
            print(f"Navegando na página {page_num}...")
            response = requests.get(page_url)
            soup = BeautifulSoup(response.text, 'html.parser')
            produtos = soup.find_all('li', class_='product-layout')

            if not produtos:
                break

            for produto in produtos:
                dados_produto = extrair_dados_produto(produto)
                if dados_produto:
                    dados_produtos.append(dados_produto)

            page_num += 1

        # Salvar os dados em CSV
        if dados_produtos:
            salvar_dados_csv(dados_produtos, category)

    except requests.RequestException as e:
        print(f"\033[91mErro ao fazer a requisição para a página {url}: {e}\033[0m")
    except Exception as e:
        print(f"\033[91mErro ao extrair dados da página {url}: {e}\033[0m")

# Função principal
def main():
    try:
        # Lista de links da Fogo E Shop com a categoria
        links = {
            "Cabine": "https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/acessorios-banheiro/acessorio-cabide/"
            "Chuveiro Eletrico": "https://fogoesshop.com.br/fshop-banho/chuveiro-eletrico/",
        }

        # Iterar sobre as categorias e extrair dados
        for category, url in links.items():
            extrair_dados(url, category)

    except Exception as e:
        print(f"\033[91mErro no processo principal: {e}\033[0m")

if __name__ == "__main__":
    main()


Processando categoria 'Cabine' - URL: https://fogoesshop.com.br/acessorios-banheiros-portatoalha-papeleira-prateleira-cabide/acessorios-banheiro/acessorio-cabide/
Navegando na página 1...
Navegando na página 2...
Navegando na página 3...
Navegando na página 4...
Dados da categoria 'Cabine' salvado com sucesso

ARQUIVO CRIADO 2024-05-16
